# 07 Shuffle

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Explain step by step what happens in a shuffle (map side, shuffle files, network fetch, reduce side), why it is expensive, how <code>spark.sql.shuffle.partitions</code> and AQE shape it, and the practical ways to shuffle less data.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"How does a shuffle work and why is it expensive?" is one of the six spoken answers this learning path asks you to prepare. Shuffles dominate the cost of joins, aggregations, deduplication and sorting, and most tuning (partition counts, skew handling, broadcast joins, pre-aggregation) is about the shuffle.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The shuffle in four steps</b>
<ol><li><b>Map side</b>: each task of the stage before the shuffle computes a target partition for every row (for example <code>hash(key) % N</code>), sorts or buffers rows by target, and <b>writes shuffle files to local disk</b>, one data file plus an index per task</li><li><b>Stage boundary</b>: the next stage can't start until all map tasks have finished</li><li><b>Fetch</b>: each reduce task pulls its blocks from <b>every</b> map task's files, across the network</li><li><b>Reduce side</b>: the task merges the blocks (often sorting or hashing them) and continues with the next operator (final aggregate, join, ...)</li></ol>
Why it's expensive: <b>serialization</b>, <b>disk writes and reads</b>, <b>network transfer</b>, <b>memory pressure</b> (which can cause spill to disk), and a <b>barrier</b> between stages.
</div>

```
Stage 1 (map, 4 tasks)                          Stage 2 (reduce, 3 tasks)
 task 1 ─write─▶ [blocks for R1,R2,R3] ─┐
 task 2 ─write─▶ [blocks for R1,R2,R3] ─┼─ network ─▶ R1 fetches its block from all 4 map outputs
 task 3 ─write─▶ [blocks for R1,R2,R3] ─┤             R2 ...
 task 4 ─write─▶ [blocks for R1,R2,R3] ─┘             R3 ...
              local disk                    4 × 3 = 12 blocks moved
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A deduplication job on 2 TB of events took 3 hours. The query profile showed 1.8 TB of shuffle write, because the job selected all 120 columns before <code>dropDuplicates(["event_id"])</code>. Selecting the 15 columns actually needed <b>before</b> the deduplication cut the shuffle to 250 GB and the runtime to 30 minutes. Same logic, less data moved.
</div>

## Syntax: the knobs

| Setting / tool | Effect | Serverless |
|---|---|---|
| `spark.sql.shuffle.partitions` | Number of reduce partitions (default 200) | Settable (often `auto`) |
| AQE coalescing (`spark.sql.adaptive.coalescePartitions.enabled`) | Merges small shuffle partitions at runtime | On, managed |
| `F.broadcast(df)` / join hints | Avoid shuffling the big side of a join | Works |
| Filter and select before wide operations | Less data to shuffle | Works |
| Query profile / Spark UI "Shuffle Read / Write" | Measure shuffle size | Query profile |

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates helpers and a sample "events" DataFrame with many columns.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output.
</div>

In [0]:
import time, io, contextlib, re
from pyspark.sql import functions as F

def timed(label, fn):
    start = time.time()
    result = fn()
    print(f"{label:<45} {time.time() - start:6.2f}s")
    return result

def plan_text(df):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

events = (
    spark.range(0, 3_000_000, 1, numPartitions=8)
    .select(
        F.col("id").alias("event_id"),
        (F.col("id") % 100_000).alias("user_id"),
        (F.col("id") % 50).alias("page"),
        *[F.concat(F.lit(f"attr{i}_"), (F.col("id") % 1000).cast("string")).alias(f"attr{i}") for i in range(12)],
    )
)

## 1. Cost of a shuffle, measured

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs a warm-up query, then times a narrow-only query and two queries with a shuffle. Each result is written to the <code>noop</code> format, which executes the whole plan and discards the output.<br><br>
<b>Why it matters</b><br>Seeing the gap on your own compute makes the cost concrete. The narrow query reads and filters. The shuffle queries also have to write, transfer and read rows by key. <code>count()</code> is a poor benchmark, because the optimizer can skip work whose result isn't needed for a count.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The narrow query is the fastest. Deduplicating while keeping all 15 columns is clearly slower than deduplicating a single column. Exact times depend on the compute.
</div>

In [0]:
noop = lambda df: df.write.format("noop").mode("overwrite").save()

noop(events.limit(1000))                                            # warm-up, so the first timing isn't distorted
timed("narrow only (filter)", lambda: noop(events.filter("page < 25")))
timed("shuffle 1 column (distinct users)", lambda: noop(events.select("user_id").distinct()))
timed("shuffle 15 columns (dropDuplicates)", lambda: noop(events.dropDuplicates(["user_id"])))

## 2. Shuffle less data: keep only the columns you need

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Deduplicates users and writes the result, once keeping all 15 columns and once keeping only the 2 columns a consumer needs.<br><br>
<b>Why it matters</b><br>A shuffle moves <b>every column of every row</b> that reaches it. Spark already prunes columns that nothing uses later. But if you keep, write or return columns you don't need, they travel through every shuffle. Select the columns you need, and filter the rows you need, before the wide operation.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The 2-column version is faster. Both produce the same number of users.
</div>

In [0]:
timed("dedupe, keep 15 columns", lambda: noop(events.dropDuplicates(["user_id"])))
timed("select 2 columns, then dedupe", lambda: noop(events.select("user_id", "page").dropDuplicates(["user_id"])))
print("same user count:",
      events.dropDuplicates(["user_id"]).count() == events.select("user_id", "page").dropDuplicates(["user_id"]).count())

## 3. Partial aggregation shrinks the shuffle

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the two <code>HashAggregate</code> steps around the exchange, and compares the input rows with the rows that actually cross the shuffle.<br><br>
<b>Why it matters</b><br>For <code>groupBy().agg()</code>, Spark aggregates <b>inside each map task first</b> (like a MapReduce combiner), so only one row per key per task is shuffled. That's why <code>groupBy("page").count()</code> on 3 million rows shuffles at most <code>50 pages × 8 tasks</code> rows. Functions that can't be partially aggregated well (for example <code>collect_list</code>) shuffle much more.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A plan with <code>partial_count</code> before the <code>Exchange</code> and <code>count</code> after it, then the maximum number of rows crossing the shuffle: 400.
</div>

In [0]:
by_page = events.groupBy("page").count()
for line in plan_text(by_page).splitlines():
    if "HashAggregate" in line or "Exchange" in line:
        print(line.strip()[:120])

map_tasks, keys = 8, 50
print(f"input rows: {events.count():,} | rows crossing the shuffle: at most {map_tasks * keys}")

## 4. Shuffle partitions: too few, too many

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs the same aggregation with 4, 64 and 1,000 shuffle partitions, with AQE coalescing disabled where possible to see the raw effect.<br><br>
<b>Why it matters</b><br>Too <b>few</b> partitions means big tasks that can run out of memory and spill, and idle cores. Too <b>many</b> means tiny tasks whose scheduling overhead dominates. A common target is shuffle partitions of roughly 100–200 MB each. AQE fixes "too many" automatically by coalescing.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
On serverless, <code>spark.sql.shuffle.partitions</code> can usually be set, but AQE settings may be locked. If the AQE setting is rejected, the timings show AQE coalescing at work and will be similar.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The 1,000-partition run is usually the slowest on small data (too much overhead). The partition count of the result matches the setting when AQE is off.
</div>

In [0]:
original = spark.conf.get("spark.sql.shuffle.partitions")
try:
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    aqe = "off"
except Exception:
    aqe = "managed (could not change)"
print("AQE:", aqe)

query = events.groupBy("user_id").agg(F.count("*").alias("n"))
for n in [4, 64, 1000]:
    spark.conf.set("spark.sql.shuffle.partitions", str(n))
    timed(f"shuffle.partitions = {n}", lambda: query.write.format("noop").mode("overwrite").save())

spark.conf.set("spark.sql.shuffle.partitions", original)
try:
    spark.conf.set("spark.sql.adaptive.enabled", "true")
except Exception:
    pass

## 5. AQE coalesces small shuffle partitions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Groups by a column with only 10 values, with 200 shuffle partitions, and counts how many partitions actually hold data after AQE.<br><br>
<b>Why it matters</b><br>Without AQE, 200 reduce tasks would run, 190 of them empty. AQE looks at the real shuffle sizes after the map stage and merges small partitions, so far fewer tasks run. This is why the 200 default rarely hurts on Databricks.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
With AQE on, the 10 groups end up in 1 (or very few) partitions instead of up to 10 of the 200.
</div>

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "200")
tiny_groups = events.groupBy((F.col("page") % 10).alias("g")).count()
print("partitions holding data:", tiny_groups.select(F.spark_partition_id()).distinct().count())
spark.conf.set("spark.sql.shuffle.partitions", original)

## 6. Avoid unnecessary shuffles (and what the optimizer removes for you)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds the same aggregation three ways: clean, with a redundant <code>repartition</code> before it, and with a redundant <code>orderBy</code> before it. Then it counts the shuffles in each plan.<br><br>
<b>Why it matters</b><br>A <code>repartition</code> before a <code>groupBy</code> on another key is a wasted shuffle, and Spark keeps it because you asked for it explicitly. A global sort before an aggregation is also pointless, and here the optimizer <b>removes</b> it for you (rule <code>EliminateSorts</code>). Knowing which waste the optimizer removes and which it can't is part of reading plans well.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Clean: 1 shuffle. With <code>repartition(100)</code>: 2 shuffles. With <code>orderBy</code> first: still 1 shuffle, because the useless sort was eliminated. All three give the same result.
</div>

In [0]:
def count_shuffles(df):
    return len(re.findall(r"(?<!Broadcast)Exchange (hash|range|RoundRobin|SinglePartition)", plan_text(df)))

agg = lambda df: df.groupBy("page").agg(F.sum("event_id").alias("total"))
variants = {
    "clean": agg(events),
    "repartition(100) first": agg(events.repartition(100)),
    "orderBy first": agg(events.orderBy("user_id")),
}
for name, df in variants.items():
    print(f"{name:<25} shuffles: {count_shuffles(df)}")
results = [sorted(df.collect()) for df in variants.values()]
print("same result:", results[0] == results[1] == results[2])

## Under the hood

```
Map task (per input partition)                 Reduce task (per shuffle partition)
  rows ─▶ compute partition id                   fetch blocks from every map output
       ─▶ buffer in memory, sort by partition id    (local or remote, over the network)
       ─▶ spill to disk if memory fills            merge / sort / hash the blocks
       ─▶ write ONE data file + index file      ─▶ continue the plan (final agg, join ...)
  map output registered with the driver
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> every shuffle writes to disk and crosses the network.

**Spill**: when a task's in-memory buffers are full, it writes sorted runs to disk and merges them later. "Spill (memory)" and "Spill (disk)" in the Spark UI or query profile are a sign that partitions are too big. More shuffle partitions, or less data per row, fixes it.

**Monitoring**: the key numbers are **shuffle write** (map side) and **shuffle read** (reduce side) bytes, and spill. On serverless, open the query profile and look at the exchange operators.

In [0]:
events.groupBy("page").agg(F.countDistinct("user_id")).explain("formatted")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: huge shuffle write in the query profile</b><br>
Too many columns or rows reach the wide operation. Select and filter first.<br><br>
<b>⛔ Problem: spill to disk, out-of-memory in reduce tasks</b><br>Shuffle partitions too large. Increase <code>spark.sql.shuffle.partitions</code>, or check for skew on one key.<br><br>
<b>⛔ Problem: thousands of tiny reduce tasks</b><br>Too many shuffle partitions for the data size. Rely on AQE coalescing, or lower the setting.<br><br>
<b>⛔ Problem: one reduce task far slower than the others</b><br>Skewed key. See AQE skew join (lesson 13) and <code>05_optimization/02_data_skew</code>.<br><br>
<b>⛔ Problem: shuffles you didn't expect</b><br>Look for redundant <code>repartition</code>, <code>orderBy</code>, <code>distinct</code>, and non-broadcast joins in <code>explain()</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is a shuffle?</b><br>
The redistribution of data across partitions so that rows with the same key end up in the same partition. It's needed by wide transformations such as <code>groupBy</code>, <code>join</code>, <code>distinct</code> and <code>orderBy</code>, and it creates a stage boundary.<br><br>

<b>🎤 2. How does a shuffle work internally?</b><br>
Each map task computes the target partition for every row, buffers and sorts by partition, and writes a data file and an index file to local disk. After all map tasks finish, each reduce task fetches its blocks from every map output over the network, merges them, and continues processing.<br><br>

<b>🎤 3. Why is a shuffle expensive?</b><br>
It serializes data, writes it to disk, transfers it over the network, reads and deserializes it again, uses memory that can cause spill, and forces a barrier between stages.<br><br>

<b>🎤 4. How do you reduce shuffle cost?</b><br>
Filter and select early, broadcast small join sides, avoid unnecessary repartition, orderBy and distinct, rely on partial aggregation, choose a sensible number of shuffle partitions, and handle skewed keys.<br><br>

<b>🎤 5. What does <code>spark.sql.shuffle.partitions</code> control?</b><br>
The number of partitions after a shuffle in DataFrame and SQL operations, 200 by default. With AQE, Spark coalesces small partitions at runtime, so the actual number is often lower.<br><br>

<b>🎤 6. What is spill?</b><br>
When a task's data doesn't fit in its execution memory, Spark writes intermediate data to disk and merges it later. It's slower and usually means partitions are too large or skewed.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A query profile shows a very large shuffle write for a <code>dropDuplicates(["order_id"])</code> step on a 150-column table. What is the most effective first fix?</b><br>
A. Increase the cluster size<br>
B. Select only the columns needed before <code>dropDuplicates</code><br>
C. Add <code>repartition(1000)</code> before the step<br>
D. Convert the table to CSV<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Less data per row means less data written and moved during the shuffle.</details><br><br>

<b>Q2. Which feature merges many small shuffle partitions into fewer, larger ones at runtime?</b><br>
A. Delta caching<br>
B. Adaptive Query Execution<br>
C. Z-ordering<br>
D. Photon<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Count distinct users per page in two ways (<code>countDistinct</code> vs <code>select(...).distinct()</code> then <code>groupBy().count()</code>) and compare their shuffle counts and times</li><li>Find the number of shuffle partitions that is fastest for <code>events.groupBy("user_id").count()</code> on your compute among 8, 32, 128, 512 (AQE off if allowed)</li><li>Rewrite <code>events.repartition(50).groupBy("page").agg(F.max("event_id"))</code> with one shuffle less</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Two ways to count distinct users per page
a = events.groupBy("page").agg(F.countDistinct("user_id").alias("users"))
b = events.select("page", "user_id").distinct().groupBy("page").count()
print("shuffles:", count_shuffles(a), "vs", count_shuffles(b))
timed("countDistinct", lambda: a.collect())
timed("distinct + count", lambda: b.collect())

# 2. Try several shuffle partition counts
try:
    spark.conf.set("spark.sql.adaptive.enabled", "false")
except Exception:
    pass
for n in [8, 32, 128, 512]:
    spark.conf.set("spark.sql.shuffle.partitions", str(n))
    timed(f"partitions={n}", lambda: events.groupBy("user_id").count().write.format("noop").mode("overwrite").save())
spark.conf.set("spark.sql.shuffle.partitions", original)
try:
    spark.conf.set("spark.sql.adaptive.enabled", "true")
except Exception:
    pass

# 3. The repartition is redundant: groupBy shuffles by page anyway
before = events.repartition(50).groupBy("page").agg(F.max("event_id"))
after = events.groupBy("page").agg(F.max("event_id"))
print("before:", count_shuffles(before), "| after:", count_shuffles(after))

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Shuffle = map tasks write partitioned files to local disk, reduce tasks fetch them over the network</li><li>Expensive: serialization, disk, network, memory (spill), and a stage barrier</li><li>Partial aggregation shrinks shuffles for <code>groupBy().agg()</code></li><li>Shuffle less: select and filter first, broadcast small sides, remove redundant wide operations</li><li><code>spark.sql.shuffle.partitions</code> (default 200) sets reduce partitions. AQE coalesces small ones</li><li>Watch shuffle read/write and spill in the query profile or Spark UI</li></ul>
</div>

| Goal | How |
|---|---|
| Less data shuffled | `select` / `filter` before wide operations |
| No shuffle of the big side | `F.broadcast(small_df)` |
| Reduce partition count | `spark.conf.set("spark.sql.shuffle.partitions", "64")` |
| Measure | Query profile → exchange nodes, shuffle bytes, spill |
| Benchmark without writing | `df.write.format("noop").mode("overwrite").save()` |

## Git commit

```
git add 02_spark_internals/07_shuffle.ipynb
git commit -m "add 02_07 shuffle notebook"
```